# Ingestão de Dados - Camada 01_RAW

Este notebook tem como objetivo realizar a ingestão dos dados brutos no ambiente Databricks, salvando-os em formato Delta na camada `01_raw`.

##  Fontes de dados
- vendas_2023_2024.csv
- produtos_raw.csv
- clientes_crm.json
- custos_importacao.json

##  Objetivo
- Ler arquivos armazenados no Volume (`raw_files`)
- Inferir schema automaticamente
- Persistir como tabelas Delta no catálogo `lh_nautical`
- Garantir reprodutibilidade e rastreabilidade dos dados

In [0]:
# Importação das bibliotecas necessárias
from pyspark.sql import SparkSession


spark = SparkSession.builder.getOrCreate()

In [0]:
# Caminho base do volume onde os arquivos foram carregados
BASE_PATH = "/Volumes/lh_nautical/01_raw/raw_files/"

# Dicionário com os caminhos dos arquivos
paths = {
    "vendas": BASE_PATH + "vendas_2023_2024.csv",
    "produtos": BASE_PATH + "produtos_raw.csv",
    "clientes": BASE_PATH + "clientes_crm.json",
    "importacao": BASE_PATH + "custos_importacao.json"
}

paths

{'vendas': '/Volumes/lh_nautical/01_raw/raw_files/vendas_2023_2024.csv',
 'produtos': '/Volumes/lh_nautical/01_raw/raw_files/produtos_raw.csv',
 'clientes': '/Volumes/lh_nautical/01_raw/raw_files/clientes_crm.json',
 'importacao': '/Volumes/lh_nautical/01_raw/raw_files/custos_importacao.json'}

In [0]:
# Leitura do arquivo CSV de vendas
df_vendas = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(paths["vendas"])
)

# Escrita como tabela Delta na camada RAW
df_vendas.write \
    .mode("overwrite") \
    .format("delta") \
    .saveAsTable("lh_nautical.01_raw.vendas_raw")

# Validação 
print(f"Total de registros em vendas_raw: {df_vendas.count()}")
df_vendas.show(5)

Total de registros em vendas_raw: 9895
+---+---------+----------+---+--------+----------+
| id|id_client|id_product|qtd|   total| sale_date|
+---+---------+----------+---+--------+----------+
|  0|       42|       105| 11|  3405.0|2023-09-10|
|  1|        3|       136|  9| 16873.9|15-09-2024|
|  2|       25|       139|  7|  9475.3|2024-08-13|
|  4|       20|        23|  5| 55893.0|2023-02-03|
|  5|        8|        57|  4|451403.9|2024-02-12|
+---+---------+----------+---+--------+----------+
only showing top 5 rows


In [0]:
# Leitura do arquivo CSV de produtos
df_produtos = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(paths["produtos"])
)

# Escrita como tabela Delta
df_produtos.write \
    .mode("overwrite") \
    .format("delta") \
    .saveAsTable("lh_nautical.01_raw.produtos_raw")

# Validação
print(f"Total de registros em produtos_raw: {df_produtos.count()}")
df_produtos.show(5)

Total de registros em produtos_raw: 157
+--------------------+-----------+----+--------------------+
|                name|      price|code|     actual_category|
+--------------------+-----------+----+--------------------+
|Transponder AIS M...|R$ 33122.52|   1|         ELETRONICOS|
|Transponder Furun...|R$ 13998.15|   2|         ELETRONICOS|
|Radar Furuno Puls...| R$ 9024.19|   3|E L E T R Ô N I C...|
|Rádio AIS Hydro T...| R$ 3381.88|   4|         Eletrunicos|
|Piloto Automático...|R$ 23669.01|   5|         Eletronicoz|
+--------------------+-----------+----+--------------------+
only showing top 5 rows


In [0]:
# Leitura do JSON de clientes (multiline = True por ser JSON estruturado)
df_clientes = (
    spark.read
    .option("multiline", True)
    .json(paths["clientes"])
)

# Escrita como tabela Delta
df_clientes.write \
    .mode("overwrite") \
    .format("delta") \
    .saveAsTable("lh_nautical.01_raw.clientes_raw")

# Validação
print(f"Total de registros em clientes_raw: {df_clientes.count()}")
df_clientes.show(5)

Total de registros em clientes_raw: 49
+----+--------------------+--------------------+--------------------+
|code|               email|           full_name|            location|
+----+--------------------+--------------------+--------------------+
|   1|femininos.oliveir...|Femininos Oliveir...|Aratu (Candeias) ...|
|   2|nunes.fernanda.so...|Fernanda Azevedo ...|         PE , Recife|
|   3|farias.teixeira.d...|Daniel Farias Rib...|       Rio Grande,RS|
|   4|thiago.moreira#gm...|      Thiago Moreira|     AC , Rio Branco|
|   5|pedro.freitas#icl...|       Pedro Freitas|  PA - Santarém Novo|
+----+--------------------+--------------------+--------------------+
only showing top 5 rows


In [0]:
# Leitura do JSON de custos de importação
df_importacao = (
    spark.read
    .option("multiline", True)
    .json(paths["importacao"])
)

# Escrita como tabela Delta
df_importacao.write \
    .mode("overwrite") \
    .format("delta") \
    .saveAsTable("lh_nautical.01_raw.custos_importacao_raw")

# Validação
print(f"Total de registros em custos_importacao_raw: {df_importacao.count()}")
df_importacao.show(5)

Total de registros em custos_importacao_raw: 150
+-----------+--------------------+----------+--------------------+
|   category|       historic_data|product_id|        product_name|
+-----------+--------------------+----------+--------------------+
|eletrônicos|[{10/08/2016, 105...|         1|Transponder AIS M...|
|eletrônicos|[{23/11/2017, 432...|         2|Transponder Furun...|
|eletrônicos|[{12/04/2016, 254...|         3|Radar Furuno Puls...|
|eletrônicos|[{04/03/2016, 909...|         4|Rádio AIS Hydro T...|
|eletrônicos|[{10/02/2016, 600...|         5|Piloto Automático...|
+-----------+--------------------+----------+--------------------+
only showing top 5 rows


In [0]:
# Verificação rápida das tabelas criadas
spark.sql("SELECT COUNT(*) AS total FROM lh_nautical.01_raw.vendas_raw").show()
spark.sql("SELECT COUNT(*) AS total FROM lh_nautical.01_raw.produtos_raw").show()
spark.sql("SELECT COUNT(*) AS total FROM lh_nautical.01_raw.clientes_raw").show()
spark.sql("SELECT COUNT(*) AS total FROM lh_nautical.01_raw.custos_importacao_raw").show()

+-----+
|total|
+-----+
| 9895|
+-----+

+-----+
|total|
+-----+
|  157|
+-----+

+-----+
|total|
+-----+
|   49|
+-----+

+-----+
|total|
+-----+
|  150|
+-----+



In [0]:
df_importacao.selectExpr("explode(historic_data)").count()

1260